# D · profile_with_embedding: examiner RAG with B's BGE-M3 search

Tasks: Disposition, Remaining Rounds, Time to Next Action. Protocol: `D_trajectory/profile_with_embedding_protocol.md`.

Three ways to give GPT-5-mini this examiner's past, each for the case's own examiner and for a wrong examiner of the
same art unit (control). Everything else in the prompt (case record, A's state, prompt text) is the same as before.

| Setup | What the LLM sees | Output tag |
| --- | --- | --- |
| **card** | **statistics only**, computed from all of the examiner's past applications before the cutoff | `gpt-5-mini_state_profile_with_embedding_card_{own,mismatched}` |
| **paths** | the **3 past applications most similar** to this case (B's BGE-M3 search), dated events only | `gpt-5-mini_state_profile_with_embedding_paths_{own,mismatched}` |
| **fulldoc** | the same 3 past applications **in full**: abstract, claims, every letter and reply before the cutoff, and their dated events | `gpt-5-mini_state_profile_with_embedding_fulldoc_{own,mismatched}` |

All three blocks are built in Step 1 into one frozen file (`inputs/examiner_blocks_emb_v1.jsonl`, hash checked).
**A setup whose results already exist is not run again:** Step 3 first looks for its finished output (100 cases, all
answered, frozen prompt, this blocks file) and skips it if found. Otherwise it runs, and any case already answered
comes from the cache.

The top-3 average (no LLM) is computed in Step 5 for free.

**My Drive must hold** (shortcuts are fine: right-click the file in the team folder → **Organize → Add shortcut → My Drive**):
- the `D_trajectory` folder (with this week's code);
- `data_v1_share_v1.2.zip`;
- B's `B_shared_BGE_M3_dense_v1_2026_11_05_TEAM_ONLY.zip` and A's `profile_corpus_v2.zip` (the three blocks are built from them).

**How to run**
1. Step 0: first time `STAGE = 'formatcheck'` (7 cases including the two longest prompts, 84 calls, about $1).
2. Secrets (key icon, Notebook access on): `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_DEPLOYMENT` (or `OPENAI_API_KEY`).
3. **Runtime → Run all.** If the format check passes, set `STAGE = 'full'` and **Run all** again.
   The full stage runs card, paths and fulldoc: 1,200 calls, about $20, several hours (the fulldoc prompts are long).
   Keep the tab open. If Colab disconnects, just **Run all** again: finished cases come from the cache.

In [7]:
# Step 0: choices
RUN_GPT = True
STAGE = 'full'   # 'formatcheck' = 7 fixed cases (protocol section 8), 'full' = all 100 test_core cases
SETUPS = ['card', 'paths', 'fulldoc']   # the three LLM setups; any with finished results is skipped in Step 3
SETTINGS = ['own', 'mismatched']
CHECK_IDS = '13060040,13484627,13627444,13577408,14031839,14506249,14940084'

In [8]:
# Step 1: find the files on Drive, unzip, build the three examiner blocks, check hashes, install the tools
from google.colab import drive
drive.mount('/content/drive')
import glob, hashlib, json, os, re, subprocess, sys
def find(name):
    return (glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=True) +
            glob.glob(f'/content/drive/.shortcut-targets-by-id/**/{name}', recursive=True))
roots = glob.glob('/content/drive/MyDrive/**/D_trajectory/code/run_trajectory_llm.py', recursive=True)
zips = find('data_v1_share_v1.2.zip')
assert roots, 'Cannot find the D_trajectory folder in My Drive.'
assert zips, 'Cannot find data_v1_share_v1.2.zip in My Drive.'
ROOT = os.path.dirname(os.path.dirname(roots[0]))
need = ['code/examiner_history.py', 'code/build_examiner_histories.py', 'code/build_examiner_blocks_emb.py', 'code/history_math.py',
        'code/topk_math.py', 'code/compare_profile_with_embedding.py', 'code/run_trajectory_llm.py', 'code/run_rounds_next_llm.py',
        'code/score_trajectory.py', 'code/score_rounds_next.py', 'code/traj_common.py',
        'prompts/trajectory_forecast_profile.txt', 'prompts/rounds_next_forecast_profile.txt',
        'inputs/state_predictions_A.jsonl', 'inputs/trajectory_inputs.jsonl', 'profile_with_embedding_protocol.md',
        'outputs/D/trajectory_predictions_km.jsonl', 'outputs/D/rounds_next_predictions_km.jsonl']
missing = [n for n in need if not os.path.exists(os.path.join(ROOT, n))]
assert not missing, f'Missing in D_trajectory on Drive: {missing}'

# frozen hashes (protocol section 4)
B_ZIP_SHA256 = 'ff3db404a413b1f1d74b3f5c1b2b4c198d9ab87ef28196cfc7c32c3496e38917'
PROMPTS = {'trajectory_forecast_profile.txt': 'd7e75c380ac8', 'rounds_next_forecast_profile.txt': '8656a89b9b39'}
BLOCKS_SHA1 = '963c00122fe59b8799e8214634b8899900977558'
for f, h in PROMPTS.items():
    g = hashlib.sha1(open(os.path.join(ROOT, 'prompts', f), 'rb').read()).hexdigest()[:12]
    assert g == h, f'Prompt {f} changed ({g}, frozen {h}). Do not edit prompts after the protocol.'

DATA = '/content/data_v1_share'
if not os.path.exists(DATA + '/labels.csv'):
    subprocess.run(['unzip', '-q', '-o', zips[0], '-d', '/content/'], check=True)
BLOCKS = os.path.join(ROOT, 'inputs', 'examiner_blocks_emb_v1.jsonl')
ok = os.path.exists(BLOCKS) and hashlib.sha1(open(BLOCKS, 'rb').read()).hexdigest() == BLOCKS_SHA1
if not ok:
    bzips, corp = find('B_shared_BGE_M3_dense_v1_2026_11_05_TEAM_ONLY.zip'), find('profile_corpus_v2.zip')
    assert bzips, "Cannot find B's B_shared_BGE_M3_dense_v1_2026_11_05_TEAM_ONLY.zip. In Drive: right-click it in the team folder → Organize → Add shortcut → My Drive."
    assert corp, 'Cannot find profile_corpus_v2.zip. In Drive: right-click it in the team folder → Organize → Add shortcut → My Drive.'
    got = hashlib.sha256(open(bzips[0], 'rb').read()).hexdigest()
    assert got == B_ZIP_SHA256, f"B's package is not the frozen BGE-M3 v1 file (sha256 {got})."
    subprocess.run(['unzip', '-q', '-o', bzips[0], '-d', '/content/bpkg'], check=True)
    BPKG = glob.glob('/content/bpkg/**/RELEASE_MANIFEST.json', recursive=True)[0].rsplit('/', 1)[0]
    subprocess.run([sys.executable, os.path.join(ROOT, 'code', 'build_examiner_blocks_emb.py'), '--bpkg', BPKG,
                    '--corpus', corp[0], '--out', BLOCKS], check=True)
got = hashlib.sha1(open(BLOCKS, 'rb').read()).hexdigest()
assert got == BLOCKS_SHA1, f'examiner_blocks_emb_v1.jsonl differs from the frozen file ({got}). Send this to Claude.'
STATE = os.path.join(ROOT, 'inputs', 'state_predictions_A.jsonl')
print('ROOT =', ROOT); print('DATA =', DATA)
!pip -q install "openai==2.54.0" "scikit-survival==0.28.0"
%cd {ROOT}/code
print('Step 1 OK')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROOT = /content/drive/MyDrive/D_trajectory
DATA = /content/data_v1_share
/content/drive/MyDrive/D_trajectory/code
Step 1 OK


In [9]:
# Step 2: read the secrets (never typed into the notebook) and test the connection
from google.colab import userdata
def get(k):
    try:
        return (userdata.get(k) or '').strip()
    except Exception:
        return ''
def sh(args):
    p = subprocess.Popen([sys.executable, '-u'] + args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    out = []
    for line in p.stdout:
        print(line, end=''); out.append(line)
    p.wait()
    return p.returncode, ''.join(out)
if RUN_GPT:
    if get('AZURE_OPENAI_ENDPOINT') and get('AZURE_OPENAI_API_KEY'):   # same route as the earlier runs
        for k in ['AZURE_OPENAI_ENDPOINT', 'AZURE_OPENAI_API_KEY', 'AZURE_OPENAI_DEPLOYMENT']:
            assert get(k), f'Secret {k} is missing'
            os.environ[k] = get(k)
        os.environ.pop('OPENAI_API_KEY', None)
        print('GPT route: Azure, deployment', os.environ['AZURE_OPENAI_DEPLOYMENT'])
    else:
        assert get('OPENAI_API_KEY'), 'Add the Azure secrets or OPENAI_API_KEY.'
        for k in ['AZURE_OPENAI_ENDPOINT', 'AZURE_OPENAI_API_KEY']:
            os.environ.pop(k, None)
        os.environ['OPENAI_API_KEY'] = get('OPENAI_API_KEY'); os.environ['AZURE_OPENAI_DEPLOYMENT'] = 'gpt-5-mini'
        print('GPT route: OpenAI API, model gpt-5-mini (the earlier runs used Azure; note this in the report)')
    rc, out = sh(['run_trajectory_llm.py', '--provider', 'azure', '--root', ROOT, '--data', DATA, '--ping'])
    assert 'connection ok' in out, 'GPT test failed. Send the message above to Claude.'
print('Step 2 OK')

GPT route: Azure, deployment gpt-5-mini
connection ok: {"ok": true} | model: gpt-5-mini-2025-08-07
Step 2 OK


In [10]:
# Step 3: the three LLM setups x own / wrong examiner, both prompts. A setup whose results already exist is skipped.
MODEL = os.environ.get('AZURE_OPENAI_DEPLOYMENT', 'gpt-5-mini').replace('/', '_')
def tag(setup, setting):
    return f'{MODEL}_state_profile_with_embedding_{setup}_{setting}'
def finished(setup, setting):
    # True if both tasks already have all 100 test_core cases answered with the frozen prompt and this blocks file
    for task, prompt in [('trajectory', 'trajectory_forecast_profile.txt'), ('rounds_next', 'rounds_next_forecast_profile.txt')]:
        f = os.path.join(ROOT, 'outputs', 'D', f'{task}_predictions_{tag(setup, setting)}.jsonl')
        if not os.path.exists(f):
            return False
        rows = [json.loads(x) for x in open(f)]
        good = [r for r in rows if r.get('status') == 'ok' and r.get('prompt_hash') == PROMPTS[prompt]
                and r.get('blocks_file_sha1') == BLOCKS_SHA1 and r.get('examiner_input') == setup]
        if len(rows) != 100 or len(good) != 100:
            return False
    return True
RAN = []   # (setup, setting, task) run in this session; Step 4 checks exactly these
for setup in SETUPS:
    for setting in SETTINGS:
        if finished(setup, setting):
            print(f'{setup} {setting}: results already exist ({tag(setup, setting)}), skipped')
            continue
        for script, task in [('run_trajectory_llm.py', 'trajectory'), ('run_rounds_next_llm.py', 'rounds_next')]:
            if STAGE == 'formatcheck':
                extra = ['--ids', CHECK_IDS, '--workers', '1',
                         '--dump-prompts', os.path.join(ROOT, 'outputs', 'D', 'dev', f'prompts_pwe_{setup}_{setting}_{task}')]
            else:
                extra = ['--workers', '1']   # 1 at a time: stays under the Azure tokens-per-minute limit (429)
            print(f'\n=== {setup} {setting} {task} ===')
            rc, out = sh([script, '--provider', 'azure', '--root', ROOT, '--data', DATA, '--split', 'core',
                          '--state', STATE, '--state-version', 'A_v1', '--examiner-input', setup,
                          '--examiner-setting', setting, '--blocks-file', BLOCKS] + extra)
            assert rc in (0, None), f'{setup} {setting} {script} stopped. Send the message above to Claude.'
            RAN.append((setup, setting, task))
print('Step 3 OK')


=== card own trajectory ===
  10/100 done
  20/100 done
  30/100 done
  40/100 done
  50/100 done
  60/100 done
  70/100 done
  80/100 done
  90/100 done
  100/100 done
gpt-5-mini_state_profile_with_embedding_card_own [core] ok 100/100  failed 0  from cache 7  new API tokens in 4,066,835 out 103,362  est. cost of this run $1.22 (at $0.25/$2.0 per 1M)
wrote /content/drive/MyDrive/D_trajectory/outputs/D/trajectory_predictions_gpt-5-mini_state_profile_with_embedding_card_own.jsonl

=== card own rounds_next ===
  10/100 done
  20/100 done
  30/100 done
  40/100 done
  50/100 done
  60/100 done
  70/100 done
  80/100 done
  90/100 done
  100/100 done
gpt-5-mini_state_profile_with_embedding_card_own [core] ok 100/100  from cache 7  tokens in 4,073,066 out 51,638  est. $1.12
wrote /content/drive/MyDrive/D_trajectory/outputs/D/rounds_next_predictions_gpt-5-mini_state_profile_with_embedding_card_own.jsonl

=== card mismatched trajectory ===
  10/100 done
  20/100 done
  30/100 done
  40/100 do

In [11]:
# Step 4 (format check only): the pass rules of protocol section 8, for the setups run in Step 3. Accuracy is NOT looked at.
import pandas as pd
if STAGE == 'formatcheck':
    if not RAN:
        print('Nothing was run: every setup already has its results. Go to STAGE = "full".')
    rows = []
    for setup, setting, task in RAN:
        f = os.path.join(ROOT, 'outputs', 'D', 'dev', f'{task}_predictions_{tag(setup, setting)}_formatcheck.jsonl')
        for line in open(f):
            r = json.loads(line)
            rows.append(dict(setup=setup, setting=setting, task=task, case=r['sample_id'], status=r['status'],
                             finish=','.join(r.get('finish_reasons') or []), attempts=r.get('n_attempts'),
                             prompt_tokens=r.get('prompt_tokens'), error=str(r.get('error') or '')[:80]))
        d = os.path.join(ROOT, 'outputs', 'D', 'dev', f'prompts_pwe_{setup}_{setting}_{task}')
        for p in glob.glob(d + '/*.txt'):
            t = open(p, encoding='utf-8').read()
            i, j = t.find('Examiner history reference available before'), t.find('\nTask')
            assert 0 < i < j, f'Examiner block missing or misplaced in {p}'
            cut = re.search(r'after the cutoff date (\d{4}-\d{2}-\d{2})', t).group(1)
            assert 'available before ' + cut in t, f'Block cutoff differs from the case cutoff in {p}'
            block = t[i:j]
            if setup == 'card':
                assert 'Statistics for this examiner' in block, f'No statistics card in {p}'
                assert 'Past application' not in block and 'Example 1' not in block and 'episode' not in block.lower(), f'Card holds more than statistics in {p}'
            else:
                assert 'Past application' in block or 'No past applications' in block, f'No past application in {p}'
                assert 'Statistics for this examiner' not in block, f'Statistics found in {p}'
            if setup == 'paths':
                assert 'median' not in block and '%' not in block, f'Statistics found in the paths block of {p}'
            if setup == 'fulldoc':
                dates = re.findall(r'\n\[[^\]|]+\|[^\]|]+\| (\d{4}-\d{2}-\d{2}) \|', block)
                assert dates and max(dates) < cut, f'A past-application document is not before the cutoff in {p}'
    if RAN:
        T = pd.DataFrame(rows); display(T)
        bad = T[(T.status != 'ok') | T.finish.str.contains('length')]
        print('largest prompt (tokens):', int(T.prompt_tokens.max()))
        ex = glob.glob(os.path.join(ROOT, 'outputs', 'D', 'dev', 'prompts_pwe_fulldoc_own_trajectory', '13060040.txt'))
        if ex:
            t = open(ex[0], encoding='utf-8').read(); i = t.find('Examiner history reference available before')
            print('One fulldoc prompt, start of the examiner block:'); print(t[i:i + 1500])
        n_ids = len(CHECK_IDS.split(','))
        print('\nFORMAT CHECK', 'PASSED' if len(bad) == 0 and len(T) == n_ids * len(RAN) else 'NOT PASSED: see the table')

In [12]:
# Step 5 (full stage only): the top-3 average (no LLM), scores, and the fixed comparisons
if STAGE == 'full':
    sh(['topk_math.py', '--root', ROOT, '--blocks', BLOCKS])
    sh(['score_trajectory.py', '--root', ROOT])
    sh(['score_rounds_next.py', '--root', ROOT])
    sh(['compare_profile_with_embedding.py', '--root', ROOT, '--model', MODEL, '--blocks', BLOCKS])
    print('ALL DONE. Take a screenshot of this page (or download this notebook) and send it to Claude.')

emb_top3_math_own: 100 cases; usable past applications per case median 3; 0 cases fall back to KM
emb_top3_math_mismatched: 100 cases; usable past applications per case median 3; 0 cases fall back to KM
cox_ph_cutoff_features     none (Cox PH, ridge alpha=1) scored 100/100  C-idx 0.5386345053147997  IBS 0.19225359360184208
emb_top3_math_mismatched   none (math on past applications) scored 100/100  C-idx 0.5546811120196239  IBS 0.20090579362978792
emb_top3_math_own          none (math on past applications) scored 100/100  C-idx 0.6208094848732625  IBS 0.184808825051117
llm_history_only           gpt-5-mini     scored 100/100  C-idx 0.5486508585445625  IBS 0.22349694022681757
llm_state_with_history     gpt-5-mini     scored 100/100  C-idx 0.4933565004088307  IBS 0.24843204997692858
llm_state_with_history_history_mismatched gpt-5-mini     scored 100/100  C-idx 0.5184995911692559  IBS 0.212500775055305
llm_state_with_history_history_own gpt-5-mini     scored 100/100  C-idx 0.64053556827473